# Amazon: Is the Market Overpaying for Growth?

A reverse-DCF walkthrough on AMZN using `valuationengine`.

## Thesis question

Amazon trades at a market cap that requires sustained growth and margin expansion. We use the open-source `valuationengine` to answer one question precisely: **what growth rate is the market currently pricing in, and is that achievable?**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from valuationengine.data.fetcher import fetch_company
from valuationengine.core.models import Assumptions
from valuationengine.core import dcf, reverse, sensitivity, scenario

import pandas as pd
import matplotlib.pyplot as plt

## 1. Auto-fetch fundamentals

The data layer pulls fundamentals via yfinance and returns a clean `Company` object.

In [ ]:
amzn = fetch_company("AMZN")
print(f"Company: {amzn.name}")
print(f"Price: ${amzn.current_price:,.2f}")
print(f"Market cap: ${amzn.market_cap/1e9:,.1f}B")
print(f"Net debt: ${amzn.net_debt/1e9:,.1f}B")
print(f"Beta: {amzn.beta:.2f}")
print(f"Historical revenue CAGR: {amzn.historical_revenue_cagr*100:.1f}%")
print(f"Avg operating margin: {amzn.avg_operating_margin*100:.1f}%")

## 2. Base-case DCF

Default assumptions: 8% revenue growth, 20% EBIT margin, 2.5% terminal growth, WACC built from a 4.5% risk-free rate and 5.5% equity risk premium. These are deliberately middle-of-the-road; the goal is to anchor before stress-testing.

In [ ]:
result = dcf.run(amzn, Assumptions())
print(result.summary())

The intrinsic value comes out very different from the market price under base assumptions. That gap is exactly what the reverse DCF resolves: which assumption needs to move to close it.

## 3. Reverse DCF: what is the market pricing in?

Solve for the revenue growth rate that, holding everything else constant, makes the DCF match the current market cap.

In [ ]:
implied_growth = reverse.solve(amzn, Assumptions(), field="revenue_growth")
print(implied_growth["interpretation"])
print()
print(f"Implied revenue growth: {implied_growth['implied_value']*100:.2f}%")

In [ ]:
implied_margin = reverse.solve(amzn, Assumptions(), field="operating_margin", bracket=(0.01, 0.60))
print(implied_margin["interpretation"])
print()
print(f"Implied operating margin: {implied_margin['implied_value']*100:.2f}%")

Compare these implied numbers to Amazon's actual historical record. If implied growth is meaningfully higher than what the business has delivered, the market is assuming reacceleration. That is a thesis you can either believe or disbelieve, but at least now it is explicit.

## 4. Sensitivity: how fragile is the valuation?

In [ ]:
x_vals = [0.05, 0.08, 0.10, 0.12, 0.15]
y_vals = [0.10, 0.15, 0.20, 0.25, 0.30]
sens = sensitivity.run(
    amzn, Assumptions(),
    x_field="revenue_growth", x_values=x_vals,
    y_field="operating_margin", y_values=y_vals,
    output="value_per_share",
)
sens

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sens.values, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(x_vals))); ax.set_xticklabels([f"{v*100:.0f}%" for v in x_vals])
ax.set_yticks(range(len(y_vals))); ax.set_yticklabels([f"{v*100:.0f}%" for v in y_vals])
ax.set_xlabel("Revenue growth"); ax.set_ylabel("Operating margin")
ax.set_title("AMZN intrinsic value per share")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 5. Scenarios

Bull / base / bear at +/- 3 percentage points on both growth and margin.

In [ ]:
scenarios = scenario.build_bull_base_bear(Assumptions(), growth_delta=0.03, margin_delta=0.03)
results = scenario.run(amzn, scenarios)

rows = []
for name, r in results.items():
    rows.append({
        "scenario": name,
        "value_per_share": r.value_per_share,
        "current_price": amzn.current_price,
        "upside_pct": r.upside * 100,
    })
pd.DataFrame(rows)

## Takeaway

The reverse DCF tells you what the market believes. The sensitivity tells you how stable that belief is. The scenarios tell you what a reasonable range of outcomes looks like. None of them is a price target. All of them are tools for sharpening your own view.